In [ ]:
!pip -qqq install pip --progress-bar off
!pip -qqq install 'crewai[tools]'==0.28.8 --progress-bar off
!pip -qqq install langchain-groq==0.1.3 --progress-bar off
!pip install exa-py

# **EXA API**

In [ ]:
from exa_py import Exa
import requests
import os

In [ ]:
exa = Exa(api_key="")

In [ ]:
response = exa.search_and_contents('Bitcoin Cryptocurrency', summary=True)

In [ ]:
type(response)

In [ ]:
print(response)

#https://pypi.org/project/exa-py/1.0.7/

In [ ]:
for result in response.results:
    print(result.title, result.url)

# **AlphaVantage API**

In [ ]:
# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key
api_key = ""
url = 'https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol=BTC&market=EUR&apikey=demo'
r = requests.get(url)
data = r.json()

print(data)

In [ ]:
import json
pretty_json = json.dumps(data, indent=8)
print(pretty_json)

Changed market to USD, separated currency and API Key

In [ ]:
api_key = "TAFO77UQQHI2EKB0"
ticker ="BTC"
url = f"https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey={api_key}"
r = requests.get(url)
data = r.json()
pretty_json = json.dumps(data, indent=8)
print(pretty_json)

#**Tools**

## **Human Tool using LangChain**

In [ ]:
from langchain.agents import load_tools
human_tools = load_tools(["human"])

## **News Tool**

In [ ]:
from crewai_tools import tool

In [ ]:
"""Search for the latest news and provide a summary about a given query using Exa API."""

def search_tool(query: str) -> str:
    # Perform the search and fetch the results
    result = exa.search_and_contents(query, summary=True)
    # Ensure results exist before processing
    if result.results:
        news_list = []
        for item in result.results:
            # Extracting attributes directly from the Result object
            news_item = {
                "title": item.title if hasattr(item, "title") else "No Title",
                "url": item.url if hasattr(item, "url") else "#",  # URL and ID are the same
                "id": item.id if hasattr(item, "id") else "#",  # ID is same as URL
                "score": item.score if hasattr(item, "score") else "No Score",
                "published_date": item.published_date if hasattr(item, "published_date") else "Unknown Date",
                "author": item.author if hasattr(item, "author") else "Unknown Author",
                "image": item.image if hasattr(item, "image") else "No Image",
                "favicon": item.favicon if hasattr(item, "favicon") else "No Favicon",
                "summary": item.summary if hasattr(item, "summary") else "No Summary",
                "highlights": item.highlights if hasattr(item, "highlights") else "No Highlights",
                "highlight_scores": item.highlight_scores if hasattr(item, "highlight_scores") else "No Highlight Scores",
            }
            news_list.append(news_item)
        # Format the news items into a readable string
        output = []
        for news_item in news_list:
            output.append(f"Title: {news_item['title']}\nURL: {news_item['url']}\nSummary: {news_item['summary']}\n")
        return "\n".join(output)
    else:
        return "No results found."

In [ ]:
@tool("search tool")
def cryptocurrency_news_tool(ticker_symbol: str) -> str:
    """Get news for a given cryptocurrency ticker symbol"""
    return search_tool.run(ticker_symbol + " cryptocurrency")

## **Price Tool**

In [ ]:
def get_daily_closing_prices(ticker) -> pd.DataFrame:

    api_key = "Enter API key here"
    url = f"https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey={api_key}"
    response = requests.get(url)
    data = response.json()
    price_data = data["Time Series (Digital Currency Daily)"]
    daily_close_prices = {
        date: prices["4. close"] for (date, prices) in price_data.items()
    }
    df = pd.DataFrame.from_dict(daily_close_prices, orient="index", columns=["price"])
    df.index = pd.to_datetime(df.index)
    df["price"] = pd.to_numeric(df["price"])
    return df

def format_prices(price_df: pd.DataFrame) -> str:
    """Format the daily closing prices."""
    text_output = [
        f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}"
        for date, row in price_df.head(10).iterrows()
    ]
    return "\n".join(text_output)
# Fetch and format Bitcoin price data

price_df = get_daily_closing_prices("BTC")
formatted_prices = format_prices(price_df)
print("\nBitcoin Daily Closing Prices:\n", formatted_prices)

In [ ]:
@tool("price tool")
def cryptocurrency_price_tool(ticker_symbol: str) -> str:

    """Get daily closing price for a given cryptocurrency ticker symbol for the previous 60 days"""
    price_df = get_daily_closing_prices(ticker_symbol)
    text_output = []
    for date, row in price_df.head(60).iterrows():
        text_output.append(f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}")
    return "\n".join(text_output)

#**Agents**

In [ ]:
from crewai import Agent, Crew, Process, Task

In [ ]:
customer_communicator = Agent(
    role="Senior cryptocurrency customer communicator",
    goal="Find which cryptocurrency the customer is interested in",
    backstory="""You're highly experienced in communicating about cryptocurrencies
    and blockchain technology with customers and their research needs""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=human_tools,
)

news_analyst = Agent(
    role="Cryptocurrency News Analyst",
    goal="""Get news for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency news.
    You have a complete understanding of macroeconomic factors, but you specialize
    into analyzing news.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_news_tool],
)

price_analyst = Agent(
    role="Cryptocurrency Price Analyst",
    goal="""Get historical prices for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency
    historical prices. You have a complete understanding of macroeconomic factors,
    but you specialize into technical analysis based on historical prices.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_price_tool],
)

writer = Agent(
    role="Cryptocurrency Report Writer",
    goal="""Write 1 paragraph report of the cryptocurrency market.""",
    backstory="""
    You're widely accepted as the best cryptocurrency analyst that
    understands the market and have tracked every asset for more than 10 years. Your trends
    analysis are always extremely accurate.
    You're also master level analyst in the traditional markets and have deep understanding
    of human psychology. You understand macro factors and combine those multiple
    theories - e.g. cycle theory. You're able to hold multiple opinions when analysing anything.
    You understand news and historical prices, but you look at those with a
    healthy dose of skepticism. You also consider the source of news articles.
    Your most well developed talent is providing clear and concise summarization
    that explains very complex market topics in simple to understand terms.
    Some of your writing techniques include:
    - Creating a bullet list (executive summary) of the most important points
    - Distill complex analyses to their most important parts
    You writing transforms even dry and most technical texts into
    a pleasant and interesting read.""",
    llm=llm,
    verbose=True,
    max_iter=5,
    memory=True,
    allow_delegation=False,
)

#**Tasks**

In [ ]:
get_cryptocurrency = Task(
    description=f"Ask which cryptocurrency the customer is interested in.",
    expected_output="""Cryptocurrency symbol that the human wants you to research e.g. BTC.""",
    agent=customer_communicator,
)

from datetime import datetime

get_news_analysis = Task(
    description=f"""
    Use the search tool to get news for the cryptocurrency
    The current date is {datetime.now()}.
    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph report for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=news_analyst,
    context=[get_cryptocurrency],
)

get_price_analysis = Task(
    description=f"""
    Use the price tool to get historical prices
    The current date is {datetime.now()}.
    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph summary for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=price_analyst,
    context=[get_cryptocurrency],
)

write_report = Task(
    description=f"""Use the reports from the news analyst and the price analyst to
    create a report that summarizes the cryptocurrency""",
    expected_output="""1 paragraph report that summarizes the market and
    predicts the future prices (trend) for the cryptocurrency""",
    agent=writer,
    context=[get_news_analysis, get_price_analysis],
)

#**LLM: Llama3**

In [ ]:
os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
# Llama 3 with Groq for inference
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192")
system_message = "You are an experienced Machine Learning & AI Engineer."
human_message = "How to increase inference speed for a 7B LLM?"
prompt = ChatPromptTemplate.from_messages([("system", system_message), ("human", human_message)])
chain = prompt | llm
response = chain.invoke({"text": human_message})
print("\nLLM Response:\n", response.content)

#**Kicking off CrewAI Agent**

In [ ]:
crew = Crew(
    agents=[customer_communicator, price_analyst, news_analyst, writer],
    tasks=[get_cryptocurrency, get_news_analysis, get_price_analysis, write_report],
    verbose=2,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)

results = crew.kickoff()